# Stratégie de Régime de Marché — BTC / ETH

Signal adaptatif selon le régime de volatilité prédit :

| Régime | Condition | Signal |
|---|---|---|
| Calme | vol_lissée < vol_low | RSI (surachat/survente) |
| Neutre | vol_low ≤ vol_lissée ≤ vol_high | 0 (flat) |
| Tendu | vol_lissée > vol_high | CMM (tendance) |

Les seuils `vol_low` et `vol_high` sont des percentiles de la distribution des prédictions (calibrés sur la validation).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
import warnings
warnings.filterwarnings('ignore')
from scipy.stats import norm, mannwhitneyu
from itertools import product

from Backtester.Backtester import compute_backtest, plot_backtest
from prediction.settings import PAIRS, N_TARGETS, DATA_DIR, OUTPUT_DIR, CONTEXT_LENGTH

TC_BPS = 5.0

# ── Grilles D.2.3 ────────────────────────────────────────────────────
VOL_LOW_PCT_GRID  = [10, 20, 30, 40]
VOL_HIGH_PCT_GRID = [60, 70, 80, 90]
EWMA_SPAN_GRID    = [4, 8, 16]
RSI_PERIOD_GRID   = [14, 21, 28]
FAST_MA_GRID      = [8, 16, 32]
SLOW_MA_GRID      = [48, 96, 192]

# ── Paramètres par défaut ─────────────────────────────────────────────
VOL_LOW_PCT_DEFAULT  = 20
VOL_HIGH_PCT_DEFAULT = 80
EWMA_DEFAULT         = 8
RSI_PERIOD_DEFAULT   = 14
FAST_DEFAULT         = 16
SLOW_DEFAULT         = 96

print(f'Imports OK  |  Paires : {PAIRS}')

In [ ]:
def compute_rsi(prices_series, period):
    '''RSI (méthode Wilder) — retourne une Series dans [0, 100].'''
    delta = prices_series.diff()
    gains  = delta.clip(lower=0)
    losses = -delta.clip(upper=0)
    avg_g  = gains.ewm(com=period - 1, min_periods=period).mean()
    avg_l  = losses.ewm(com=period - 1, min_periods=period).mean()
    rs     = avg_g / avg_l.replace(0, np.nan)
    rsi    = 100.0 - 100.0 / (1.0 + rs)
    rsi[avg_l == 0] = 100.0
    return rsi.fillna(50)


def build_regime_positions(preds_df, prices_df, val_preds_thresh,
                            vol_low_pct, vol_high_pct, ewma_span,
                            rsi_period, fast_ma, slow_ma):
    '''Positions régime : RSI si calme, CMM si tendu, 0 sinon.'''
    vol_lissee = preds_df.abs().ewm(span=ewma_span, adjust=False).mean()
    positions  = pd.DataFrame(0.0, index=preds_df.index, columns=PAIRS)

    for pair in PAIRS:
        if pair not in prices_df.columns: continue
        vol_low  = np.percentile(val_preds_thresh[pair].dropna(), vol_low_pct)
        vol_high = np.percentile(val_preds_thresh[pair].dropna(), vol_high_pct)

        rsi_v  = compute_rsi(prices_df[pair].reindex(preds_df.index, method='ffill'), rsi_period)
        rsi_sig = pd.Series(0.0, index=preds_df.index)
        rsi_sig[rsi_v < 30]  =  1.0
        rsi_sig[rsi_v > 70]  = -1.0

        mf = prices_df[pair].reindex(preds_df.index, method='ffill').rolling(fast_ma, min_periods=1).mean()
        ms = prices_df[pair].reindex(preds_df.index, method='ffill').rolling(slow_ma, min_periods=1).mean()
        cmm_sig = pd.Series(np.where(mf.values > ms.values, 1., -1.), index=preds_df.index)

        v       = vol_lissee[pair]
        calm    = v < vol_low
        trend   = v > vol_high
        pos_p   = pd.Series(0.0, index=preds_df.index)
        pos_p[calm]  = rsi_sig[calm]
        pos_p[trend] = cmm_sig[trend]
        positions[pair] = pos_p

    return positions


def build_rsi_only(preds_df, prices_df, rsi_period):
    '''Signal RSI pur (ablation — régime calme permanent).'''
    positions = pd.DataFrame(0.0, index=preds_df.index, columns=PAIRS)
    for pair in PAIRS:
        if pair not in prices_df.columns: continue
        rsi_v = compute_rsi(prices_df[pair].reindex(preds_df.index, method='ffill'), rsi_period)
        sig   = pd.Series(0.0, index=preds_df.index)
        sig[rsi_v < 30] =  1.0
        sig[rsi_v > 70] = -1.0
        positions[pair] = sig
    return positions


def build_cmm_only(preds_df, prices_df, fast_ma, slow_ma):
    '''Signal CMM pur (ablation — régime tendu permanent).'''
    positions = pd.DataFrame(0.0, index=preds_df.index, columns=PAIRS)
    for pair in PAIRS:
        if pair not in prices_df.columns: continue
        pr  = prices_df[pair].reindex(preds_df.index, method='ffill')
        mf  = pr.rolling(fast_ma, min_periods=1).mean()
        ms  = pr.rolling(slow_ma, min_periods=1).mean()
        positions[pair] = np.where(mf.values > ms.values, 1., -1.)
    return positions


def get_metric(bt_obj, key, row=None):
    if bt_obj is None: return np.nan
    df   = bt_obj.metrics.toDataFrame()
    cols = [c for c in df.columns if key.lower() in c.lower()]
    if not cols: return np.nan
    r = row if (row and row in df.index) else df.index[0]
    try:
        return float(str(df.loc[r, cols[0]]).replace('%','').replace('bps','').strip())
    except Exception:
        return np.nan


def sharpe_fast(pos_df, perf_df, tc_bps=TC_BPS):
    common = pos_df.index.intersection(perf_df.index)
    bt = compute_backtest(
        pos_df.loc[common], perf_df.loc[common],
        transaction_costs_bps=tc_bps, compute_max_DD=False, show_pnl=False,
        position_lag_value=1, check_index=True, position_fill_value=0.0
    )
    row = 'Total' if (pos_df.ndim > 1 and pos_df.shape[1] > 1) else None
    return get_metric(bt, 'sharpe', row=row)


def extract_pnl_daily(pos_df, ret_df):
    common = pos_df.index.intersection(ret_df.index)
    pnl = (pos_df.loc[common] * ret_df.loc[common]).sum(axis=1)
    return pnl.resample('1D').sum().dropna()


def extract_trade_pnl(pos_df, ret_df, pair):
    common  = pos_df.index.intersection(ret_df.index)
    pos_arr = pos_df.loc[common, pair].values
    pnl_arr = (pos_df.loc[common, pair] * ret_df.loc[common, pair]).values
    sign_v  = np.sign(pos_arr)
    changes = np.where(np.diff(sign_v, prepend=sign_v[0]) != 0)[0]
    bounds  = np.concatenate([[0], changes, [len(pos_arr)]])
    trades  = []
    for i in range(len(bounds) - 1):
        s, e = bounds[i], bounds[i+1]
        if np.any(pos_arr[s:e] != 0): trades.append(pnl_arr[s:e].sum())
    return np.array(trades)


print('Fonctions utilitaires OK')

In [ ]:
# ── Chargement des prédictions ────────────────────────────────────────
preds = {}
for split in ('train', 'validation', 'test'):
    df = pd.read_parquet(OUTPUT_DIR / f'{split}_lstm_1l_predictions.parquet')
    df.columns = PAIRS
    preds[split] = df

train_preds = preds['train']
val_preds   = preds['validation']
test_preds  = preds['test']
all_preds   = pd.concat(list(preds.values())).sort_index()

train_start, train_end = train_preds.index[0],  train_preds.index[-1]
val_start,   val_end   = val_preds.index[0],    val_preds.index[-1]
test_start,  test_end  = test_preds.index[0],   test_preds.index[-1]

# ── Prix et rendements 15-min ─────────────────────────────────────────
prices_list, returns_list = [], []
for crypto in PAIRS:
    df1m  = pd.read_parquet(DATA_DIR / f'{crypto}_1m.parquet')
    close = df1m.loc[df1m.index.minute % 15 == 0, 'close'].sort_index()
    ret   = np.log(close / close.shift(1))
    prices_list.append(close.rename(crypto))
    returns_list.append(ret.rename(crypto))

prices_df  = pd.concat(prices_list,  axis=1).sort_index()
returns_df = pd.concat(returns_list, axis=1).sort_index()

print(f'Train : {train_start.date()} → {train_end.date()}')
print(f'Valid : {val_start.date()}   → {val_end.date()}')
print(f'Test  : {test_start.date()}  → {test_end.date()}')
print('Données chargées OK')

In [ ]:
# ── Phase 1 : RSI — vérification visuelle ─────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
pair = PAIRS[0]  # BTC

pr_val = prices_df[pair].loc[val_start:val_end]
axes[0].plot(pr_val.index, pr_val.values, color='steelblue', lw=0.7)
axes[0].set_title(f'{pair} — Prix (validation)')
axes[0].set_ylabel('Prix')
axes[0].grid(True, alpha=0.3)

rsi_val = compute_rsi(pr_val, RSI_PERIOD_DEFAULT)
axes[1].plot(rsi_val.index, rsi_val.values, color='darkorange', lw=0.7)
axes[1].axhline(30, color='green', ls='--', lw=1.2, label='RSI=30 (survente)')
axes[1].axhline(70, color='red',   ls='--', lw=1.2, label='RSI=70 (surachat)')
axes[1].axhline(50, color='gray',  ls=':',  lw=0.8)
axes[1].set_ylim(0, 100)
axes[1].set_title(f'RSI {pair} (période={RSI_PERIOD_DEFAULT} × 15min, validation)')
axes[1].set_ylabel('RSI')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
# ── Phase 2 : Backtest avec paramètres par défaut ─────────────────────
pos_default = build_regime_positions(
    test_preds, prices_df,
    val_preds_thresh=val_preds,
    vol_low_pct=VOL_LOW_PCT_DEFAULT, vol_high_pct=VOL_HIGH_PCT_DEFAULT,
    ewma_span=EWMA_DEFAULT, rsi_period=RSI_PERIOD_DEFAULT,
    fast_ma=FAST_DEFAULT, slow_ma=SLOW_DEFAULT
)

# Visualisation régimes sur BTC (test)
pair = PAIRS[0]
vol_lis = test_preds[[pair]].abs().ewm(span=EWMA_DEFAULT, adjust=False).mean()
vol_low_d  = np.percentile(val_preds[pair].dropna(), VOL_LOW_PCT_DEFAULT)
vol_high_d = np.percentile(val_preds[pair].dropna(), VOL_HIGH_PCT_DEFAULT)
regime = pd.Series('neutral', index=test_preds.index)
regime[vol_lis[pair] < vol_low_d]  = 'calm'
regime[vol_lis[pair] > vol_high_d] = 'trend'
for r in ['calm', 'neutral', 'trend']:
    print(f'{pair} test — régime {r:8s} : {(regime == r).mean():.1%}')

fig, ax = plt.subplots(figsize=(14, 4))
pr_test = prices_df[pair].reindex(test_preds.index, method='ffill')
ymin, ymax = pr_test.min(), pr_test.max()
ax.plot(pr_test.index, pr_test.values, color='steelblue', lw=0.6, zorder=3)
for r, c in [('calm','green'), ('neutral','lightgray'), ('trend','orange')]:
    mask = regime.reindex(pr_test.index) == r
    ax.fill_between(pr_test.index, ymin, ymax, where=mask, color=c, alpha=0.25, label=f'Régime {r}')
ax.set_title(f'{pair} — Prix et régimes (test, params défaut)')
ax.legend(loc='upper left'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Backtest défaut
bt_default = compute_backtest(
    pos_default.reindex(returns_df.loc[test_start:test_end].index, fill_value=0),
    returns_df.loc[test_start:test_end],
    detail_assets=True, position_lag_value=1, check_index=True, position_fill_value=0.0,
    compute_max_DD=True, transaction_costs_bps=TC_BPS
)
print('\n── Défaut (test) ───────────────────────────────────────────────')
print(bt_default.metrics.toDataFrame().to_string())

In [ ]:
# ── Phase 3 : Grid Search sur la validation ───────────────────────────
val_returns_g = returns_df.loc[val_start:val_end]

best_sharpe, best_params = -999., {}
results = []
n_skip = 0

for vol_low, vol_high in product(VOL_LOW_PCT_GRID, VOL_HIGH_PCT_GRID):
    if vol_high - vol_low < 40:  # contrainte zone neutre suffisante
        n_skip += 1; continue
    for ewma, rsi, fast, slow in product(EWMA_SPAN_GRID, RSI_PERIOD_GRID, FAST_MA_GRID, SLOW_MA_GRID):
        if fast >= slow: continue
        pos_v = build_regime_positions(
            val_preds, prices_df,
            val_preds_thresh=val_preds,
            vol_low_pct=vol_low, vol_high_pct=vol_high,
            ewma_span=ewma, rsi_period=rsi, fast_ma=fast, slow_ma=slow
        )
        sh = sharpe_fast(pos_v, val_returns_g)
        params = {'vol_low': vol_low, 'vol_high': vol_high, 'ewma': ewma,
                  'rsi': rsi, 'fast': fast, 'slow': slow}
        results.append({**params, 'sharpe': round(sh, 4)})
        if sh > best_sharpe:
            best_sharpe = sh; best_params = params

df_res = pd.DataFrame(results).sort_values('sharpe', ascending=False)
print(f'Grid search terminée ({len(results)} combinaisons, {n_skip} filtrées)')
print('\nTOP 10 (Sharpe net frais, validation) :')
print(df_res.head(10).to_string(index=False))
print(f'\nMeilleurs paramètres : {best_params}  (Sharpe = {best_sharpe:.3f})')

In [ ]:
# ── Phase 4 : Backtest best_params + Ablation RSI-only / CMM-only ─────
pos_best = build_regime_positions(
    test_preds, prices_df, val_preds_thresh=val_preds, **best_params
)

pos_rsi_only = build_rsi_only(test_preds, prices_df, best_params['rsi'])
pos_cmm_only = build_cmm_only(test_preds, prices_df, best_params['fast'], best_params['slow'])

ret_test = returns_df.loc[test_start:test_end]

bts = {}
for name, pos in [
    ('Régime D.2.3 (best)', pos_best),
    ('RSI pur',             pos_rsi_only),
    ('CMM pur',             pos_cmm_only),
]:
    bts[name] = compute_backtest(
        pos.reindex(ret_test.index, fill_value=0),
        ret_test,
        detail_assets=True, position_lag_value=1, check_index=True, position_fill_value=0.0,
        compute_max_DD=True, transaction_costs_bps=TC_BPS
    )
    sh = get_metric(bts[name], 'sharpe', row='Total')
    pnl_t = get_metric(bts[name], 'PnL/Trade', row='Total')
    print(f'{name:25s}  Sharpe={sh:+.3f}  PnL/Trade={pnl_t:+.2f} bps')

# PnL cumulé (BTC, test)
pair = PAIRS[0]
fig, ax = plt.subplots(figsize=(14, 4))
for name, pos, color in [
    ('Régime D.2.3', pos_best,     'steelblue'),
    ('RSI pur',       pos_rsi_only, 'green'),
    ('CMM pur',       pos_cmm_only, 'darkorange'),
]:
    common = pos.index.intersection(ret_test.index)
    pnl = (pos.loc[common, pair] * ret_test.loc[common, pair]).cumsum()
    pnl.plot(ax=ax, label=name, color=color, lw=1.0)
ax.axhline(0, color='black', lw=0.5, ls=':')
ax.set_title(f'{pair} — PnL cumulé (test) : Régime vs Ablations')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── Phase 5 : Analyse des Régimes ─────────────────────────────────────
print('=== Proportion du temps dans chaque régime (test) ===')
for pair in PAIRS:
    vl = test_preds[[pair]].abs().ewm(span=best_params['ewma'], adjust=False).mean()
    vlow  = np.percentile(val_preds[pair].dropna(), best_params['vol_low'])
    vhigh = np.percentile(val_preds[pair].dropna(), best_params['vol_high'])
    v = vl[pair]
    print(f'{pair}: calme={( v < vlow ).mean():.1%}  '
          f'neutre={(( v >= vlow) & (v <= vhigh)).mean():.1%}  '
          f'tendu={(v > vhigh).mean():.1%}')

# PnL par régime (BTC, test)
pair = PAIRS[0]
vl   = test_preds[[pair]].abs().ewm(span=best_params['ewma'], adjust=False).mean()
vlow  = np.percentile(val_preds[pair].dropna(), best_params['vol_low'])
vhigh = np.percentile(val_preds[pair].dropna(), best_params['vol_high'])
regime_test = pd.Series('neutral', index=test_preds.index)
regime_test[vl[pair] < vlow]  = 'calm'
regime_test[vl[pair] > vhigh] = 'trend'

pos_p  = pos_best[pair].reindex(ret_test.index).fillna(0)
ret_p  = ret_test[pair]
reg_al = regime_test.reindex(ret_test.index, method='ffill').fillna('neutral')

pos_regimes = pd.DataFrame({
    'Calme':   pos_p.where(reg_al == 'calm',    0),
    'Neutre':  pos_p.where(reg_al == 'neutral', 0),
    'Tendu':   pos_p.where(reg_al == 'trend',   0),
})
perf_regimes = pd.DataFrame({'Calme': ret_p, 'Neutre': ret_p, 'Tendu': ret_p})

bt_regimes = compute_backtest(
    pos_regimes, perf_regimes,
    detail_assets=True, show_total=True, position_lag_value=1, check_index=True,
    position_fill_value=0.0, compute_max_DD=True, transaction_costs_bps=TC_BPS,
    title=f'PnL par Régime — {pair} Test'
)
print(f'\n=== PnL par régime — {pair} (test) ===')
print(bt_regimes.metrics.toDataFrame().to_string())
plot_backtest(bt_regimes)

In [ ]:
# ── Phase 6 : Tests Statistiques (obligatoire) ────────────────────────
def newey_west_var(d, bw):
    n = len(d); nw = np.var(d, ddof=1)
    for lag in range(1, bw + 1):
        nw += 2 * (1 - lag/(bw+1)) * np.cov(d[lag:], d[:-lag])[0,1]
    return max(nw, 1e-16) / n

def dm_test(pnl1, pnl2):
    common = pnl1.index.intersection(pnl2.index)
    d = (pnl1.loc[common] - pnl2.loc[common]).dropna().values
    n = len(d)
    if n < 10: return np.nan, np.nan, n
    bw = math.ceil(n ** (1/3))
    v  = newey_west_var(d, bw)
    dm = np.mean(d) / np.sqrt(v)
    pv = 2 * (1 - norm.cdf(abs(dm)))
    return dm, pv, n

pnl_d23      = extract_pnl_daily(pos_best.reindex(ret_test.index, fill_value=0), ret_test)
pnl_rsi_only = extract_pnl_daily(pos_rsi_only.reindex(ret_test.index, fill_value=0), ret_test)
pnl_cmm_only = extract_pnl_daily(pos_cmm_only.reindex(ret_test.index, fill_value=0), ret_test)

print('=== Tests Diebold-Mariano (PnL journaliers) ===')
print(f'{"Comparaison":<30} {"DM":>8} {"p-val":>8} {"n":>6} {"Sig5%":>6}')
print('-' * 60)
for lbl, p1, p2 in [
    ('D.2.3 vs RSI pur', pnl_d23, pnl_rsi_only),
    ('D.2.3 vs CMM pur', pnl_d23, pnl_cmm_only),
]:
    dm, pv, n = dm_test(p1, p2)
    sig = 'Oui' if (not np.isnan(pv) and pv < 0.05) else 'Non'
    dm_s = f'{dm:+.3f}' if not np.isnan(dm) else 'NaN'
    pv_s = f'{pv:.3f}'  if not np.isnan(pv) else '---'
    print(f'{lbl:<30} {dm_s:>8} {pv_s:>8} {n:>6} {sig:>6}')

# Mann-Whitney par trade (BTC)
pair = PAIRS[0]
print(f'\n=== Mann-Whitney PnL/trade ({pair}) ===')
t_d23      = extract_trade_pnl(pos_best.reindex(ret_test.index,     fill_value=0), ret_test, pair)
t_rsi_only = extract_trade_pnl(pos_rsi_only.reindex(ret_test.index, fill_value=0), ret_test, pair)
t_cmm_only = extract_trade_pnl(pos_cmm_only.reindex(ret_test.index, fill_value=0), ret_test, pair)

for name, t in [('D.2.3', t_d23), ('RSI pur', t_rsi_only), ('CMM pur', t_cmm_only)]:
    med = np.median(t) * 1e4 if len(t) else np.nan
    print(f'{name:12s}: {len(t):4d} trades  médiane={med:+.2f} bps')

for (n1, t1), (n2, t2) in [
    (('D.2.3', t_d23), ('RSI pur', t_rsi_only)),
    (('D.2.3', t_d23), ('CMM pur', t_cmm_only)),
]:
    if len(t1) > 1 and len(t2) > 1:
        stat, pv = mannwhitneyu(t1, t2, alternative='two-sided')
        print(f'Mann-Whitney {n1} vs {n2} : U={stat:.0f}  p={pv:.4f}  Sig5%={"Oui" if pv < 0.05 else "Non"}')

In [ ]:
# ── Phase 7 : Tableau de Synthèse Final ───────────────────────────────
METRICS = [('Sharpe','sharpe'), ('Return/an','return (yearly)'), ('PnL/Trade','PnL/Trade'), ('Max DD','max DD')]

rows = []
for cfg, bt in [
    ('Défaut (params D.2.3)', bt_default),
    ('Best params',           bts['Régime D.2.3 (best)']),
    ('RSI pur (ablation)',    bts['RSI pur']),
    ('CMM pur (ablation)',    bts['CMM pur']),
]:
    for asset in PAIRS + ['Total']:
        r = {'Configuration': cfg, 'Actif': asset}
        for lbl, key in METRICS:
            r[lbl] = get_metric(bt, key, row=asset)
        rows.append(r)

summary = pd.DataFrame(rows).set_index(['Configuration', 'Actif'])
print('=' * 70)
print('  SYNTHÈSE — Régime D.2.3 vs Ablations (jeu de test, net frais)')
print('=' * 70)
print(summary.to_string())